# Problems with RNNs, LSTMs, and GRUs: The Need for Better Architectures

## 1. The Limitations of RNNs

Recurrent Neural Networks (RNNs) were among the earliest models designed to handle sequential data such as text, audio, or time-series. The key idea behind RNNs is that they process input one step at a time, while maintaining a hidden state that captures information from previous time steps.

However, RNNs face several critical problems:

- **Sequential Computation**:  
  RNNs must process tokens one after another. This means the model cannot take advantage of modern hardware that thrives on parallel computations. For example, if we have a sentence of 20 words, the RNN has to compute the hidden state of the 1st word before moving to the 2nd, then to the 3rd, and so on. This severely slows down training.

- **Vanishing and Exploding Gradients**:  
  During backpropagation through time (BPTT), gradients can either shrink rapidly (vanishing) or grow uncontrollably (exploding). This makes it very hard for RNNs to capture long-term dependencies. For instance, in the sentence *“The book that the professor recommended yesterday was amazing”*, the word *“book”* at the beginning is related to *“amazing”* at the end. A simple RNN often fails to preserve this dependency due to vanishing gradients.

- **Limited Long-Term Context**:  
  Because of the above issues, RNNs tend to focus more on recent tokens and forget earlier context. This is problematic in tasks like document summarization or translation, where distant tokens carry important information.

<div align="center">

 [![rnn-2-drawio.png](https://i.postimg.cc/yYQ9bHqZ/rnn-2-drawio.png)](https://postimg.cc/kBSB2LgJ)

</div>

**Sentence 2:**

John lived in Nepal for over 15 years. He listens to Nepali songs. He reads Nepali books. He is fluent in _______________.

Here RNN fails to predict it as Nepali in this context.
*We already discuss about it in  LSTM section*.

## 2. The Evolution: LSTMs and GRUs

To overcome the vanishing gradient problem, researchers introduced **Long Short-Term Memory networks (LSTMs)** and later **Gated Recurrent Units (GRUs)**. Both introduce gating mechanisms that control the flow of information.

- **LSTMs**:  
  They add an explicit memory cell and three gates (input, forget, output) to regulate which information is stored, discarded, or passed along. This helps maintain context over longer sequences compared to vanilla RNNs.

- **GRUs**:  
  A simplified version of LSTMs, with only two gates (update and reset). They achieve comparable performance with fewer parameters.

Despite these improvements, **fundamental problems remain**:

1. **Sequential Bottleneck Still Exists**:  
   LSTMs and GRUs still process tokens step by step. Even if they handle long dependencies better, they cannot parallelize sequence processing. Training on very large datasets becomes inefficient.

2. **Difficulty with Very Long Sequences**:  
   While they extend the memory compared to RNNs, their effective context window is still limited. For instance, processing entire books or long documents still causes earlier information to fade.

3. **High Computational Cost**:  
   Each step involves multiple gate computations. For example, an LSTM computes multiple matrix multiplications per token, which increases training time significantly.



## 3. Example: Sequential Bottleneck in Action

Let’s imagine we want to process the sentence:

"The quick brown fox jumps over the lazy dog"


- An **RNN/LSTM/GRU** must process this word by word:
  - Compute hidden state for "The" → pass it forward  
  - Compute hidden state for "quick" using "The" → pass it forward  
  - … and so on until "dog".

This means if the sequence has **N tokens**, the model performs **N sequential steps**. You cannot compute the hidden states for "quick" and "dog" at the same time, because "dog" depends on all previous states.

This dependency chain **prevents parallel computation**, making training and inference slow.

<div align='center'>

[![Seq2seq.png](https://i.postimg.cc/Fs7vp2YP/Seq2seq.png)](https://postimg.cc/GHw6hgJY)

*figure: Seq2Seq Model*
</div>


In traditional Seq2Seq models using **LSTM or GRU**, the encoder processes the entire input sequence and compresses it into a **single fixed-length context vector**.  
This context vector is then passed to the decoder to generate the output sequence.

However, this approach has a major limitation:

- When the input sentence is **short**, the context vector can capture most of the information successfully.  
- But for **long sentences**, all the semantic and syntactic information is squeezed into just one vector.  
- This leads to an **information bottleneck**, where important details are lost.  
- As a result, the decoder often fails to **predict words correctly** and cannot maintain **long-range semantic associations** between distant words.  

This explains why LSTM/GRU Seq2Seq models struggle with **long-range dependencies**, making translations and sequence predictions less accurate for longer inputs.



## 4. How Self-Attention and Transformers Solve This

To overcome the limitations of RNNs/LSTMs/GRUs (slow sequential processing, difficulty in handling long-range dependencies, and limited scalability), researchers introduced a new mechanism called **self-attention** and later built an architecture around it called the **Transformer**.

### What is Self-Attention?
Self-attention is a way for each token (word) in a sequence to **look at every other token** and decide how important they are for understanding its own meaning. Instead of passing information step by step (like in RNNs), self-attention creates **direct links** between tokens — no matter how far apart they are in the sequence.  
For example, in the sentence:  
*“The book that the professor recommended yesterday was amazing”*  
the word **“book”** can directly connect to **“amazing”** in a single step, without needing to go through all the intermediate words.

### Key Advantages Over RNNs
1. **Parallel Computation**  
   - In self-attention, all tokens attend to all others **at the same time**.  
   - This removes the sequential bottleneck of RNNs, making training much faster and more efficient on GPUs/TPUs.

2. **Long-Range Dependencies**  
   - Tokens can directly access information from distant tokens.  
   - Unlike RNNs, the context is not diluted over many steps.

3. **Scalability**  
   - Since computation is parallel, we can easily scale to very large datasets and models.  
   - This scalability is what enabled training massive models like **BERT, GPT, and Transformers in general**.

### What are Transformers?
Transformers are deep learning architectures that **stack multiple layers of self-attention and feed-forward networks**.  
- Each layer refines the token representations, allowing the model to capture more complex relationships.  
- Transformers also add **positional encodings** to keep track of word order, since self-attention alone does not know positions.  
- The encoder-decoder structure of the original Transformer (Vaswani et al., 2017) has since become the foundation for almost all modern NLP systems.

### In Simple Terms
- **RNNs/LSTMs**: Read words one by one (slow, forgetful for long sentences).  
- **Self-Attention**: Each word looks at all words at once (fast, remembers distant relationships).  
- **Transformers**: Powerful models built using many layers of self-attention, enabling today’s breakthroughs in NLP and beyond.



## 5. Example: Parallel Attention vs Sequential Processing

Let’s visit the sentence:  "The quick brown fox jumps over the lazy dog"


- **With RNN/LSTM/GRU**:  
  Must compute sequentially: word 1 → word 2 → … → word 9.

- **With Self-Attention**:  
  Each word computes similarity scores with *all other words* in one step.  
  - "The" attends to {quick, brown, …, dog}  
  - "dog" attends to {The, quick, …, lazy}  
  - … and so on, all in parallel.

Thus, instead of O(N) sequential steps, we perform O(1) parallel steps (though with O(N²) attention operations). This tradeoff makes training vastly faster in practice.

## 6. Summary

- **RNNs**: suffer from vanishing gradients, poor long-term memory, and sequential bottleneck.  
- **LSTMs & GRUs**: mitigate vanishing gradient but still suffer from sequential processing and limited context for very long sequences.  
- **Self-Attention & Transformers**: remove sequential bottlenecks, capture long-range dependencies efficiently, and enable parallelism, making them the backbone of modern NLP and beyond.




## *We will learn architecture details of Transformer in next notebook.*